# Week 2 Community Contribution - AdnanGobeljic

This notebook implements the Week 2 end-of-week prototype requirements:

- Gradio UI chat experience
- Streaming responses
- Expert technical system prompt
- Model switching (OpenAI + Ollama)
- Tool calling (bonus)

It is designed to stay robust with quiet runtime behavior (minimal noisy logs).

In [9]:
# imports

import os
import json
from datetime import datetime
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [10]:
# environment and model setup

load_dotenv(override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = None
if openai_api_key and openai_api_key.strip() == openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key)

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def check_ollama_running():
    try:
        response = requests.get("http://localhost:11434", timeout=2)
        return response.status_code == 200
    except Exception:
        return False

ollama_available = check_ollama_running()


In [11]:
SYSTEM_PROMPT = """
You are an expert technical tutor for software engineering, Python, and LLM engineering.
Provide practical, accurate, and concise answers.
Use markdown without wrapping your response in code fences.
When helpful, include short examples and pitfalls to avoid.
If the user asks for time/date or asks for a concept definition, use available tools.
"""


In [12]:
# bonus tools

def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def concept_lookup(concept):
    concept_map = {
        "token": "A token is a chunk of text (word piece or symbol) that the model processes as input/output units.",
        "embedding": "An embedding is a numeric vector representation of text that captures semantic meaning for search or similarity.",
        "temperature": "Temperature controls randomness in generation: lower values are more deterministic, higher values are more creative.",
        "tool calling": "Tool calling lets the model request external functions (like database lookups or APIs) during a response flow.",
        "context window": "The context window is the maximum token length the model can consider in one request.",
    }
    key = (concept or "").strip().lower()
    return concept_map.get(key, f"No local reference found for '{concept}'.")


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current local date and time.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "concept_lookup",
            "description": "Look up a quick local explanation for common LLM concepts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "concept": {
                        "type": "string",
                        "description": "Concept name, e.g. token, embedding, temperature",
                    }
                },
                "required": ["concept"],
                "additionalProperties": False,
            },
        },
    },
]


def handle_tool_calls(tool_calls):
    responses = []

    for tool_call in tool_calls:
        name = tool_call.function.name
        try:
            args = json.loads(tool_call.function.arguments or "{}")
        except Exception:
            args = {}

        if name == "get_current_time":
            content = get_current_time()
        elif name == "concept_lookup":
            content = concept_lookup(args.get("concept", ""))
        else:
            content = "Unsupported tool requested."

        responses.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": content,
            }
        )

    return responses


In [13]:
def resolve_client_and_model(model_choice):
    if model_choice == "GPT (OpenAI)":
        if openai_client is None:
            return None, None, "OpenAI client is not configured. Add OPENAI_API_KEY to .env."
        return openai_client, OPENAI_MODEL, None

    if model_choice == "Llama (Ollama)":
        if not ollama_available:
            return None, None, "Ollama is not running. Start it with: ollama serve"
        return ollama_client, OLLAMA_MODEL, None

    return None, None, "Unknown model selected."


def normalize_history(history):
    normalized = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in ("user", "assistant"):
                normalized.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, assistant_msg = item
            if user_msg:
                normalized.append({"role": "user", "content": str(user_msg)})
            if assistant_msg:
                normalized.append({"role": "assistant", "content": str(assistant_msg)})
    return normalized


def chat_stream(message, history, model_choice):
    client, model_id, error = resolve_client_and_model(model_choice)
    if error:
        yield error
        return

    normalized_history = normalize_history(history)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *normalized_history,
        {"role": "user", "content": message},
    ]

    tools_enabled = True
    try:
        response = client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
    except Exception:
        tools_enabled = False
        response = client.chat.completions.create(model=model_id, messages=messages)

    while tools_enabled and response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        messages.extend(handle_tool_calls(assistant_message.tool_calls or []))
        try:
            response = client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
        except Exception:
            tools_enabled = False
            response = client.chat.completions.create(model=model_id, messages=messages)
            break

    final_text = ""
    try:
        stream = client.chat.completions.create(
            model=model_id,
            messages=messages,
            stream=True,
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            if delta:
                final_text += delta
                yield final_text
    except Exception:
        final_text = response.choices[0].message.content or ""
        yield final_text


In [ ]:
model_selector = gr.Dropdown(
    choices=["GPT (OpenAI)", "Llama (Ollama)"],
    value="GPT (OpenAI)",
    label="Model",
)

demo = gr.ChatInterface(
    fn=chat_stream,
    additional_inputs=[model_selector],
    title="Week 2 Technical Tutor - AdnanGobeljic",
    description="Streaming technical assistant with model switching and tool calling.",
    examples=[
        ["Explain the difference between an embedding and a token.", "GPT (OpenAI)"],
        ["What time is it right now?", "GPT (OpenAI)"],
        ["How does tool calling work in chat completions?", "Llama (Ollama)"],
    ],
)


In [ ]:
demo.launch()